# 12. 텍스트 임베딩 (Text Embedding)

이번 장에서는 단어를 컴퓨터가 다룰 수 있는 **벡터로 표현하는 방법**을 다룬다. 가장 기본적인 **원-핫 인코딩(One-hot encoding)**에서 출발해, 그 한계를 넘어서기 위한 **워드 임베딩(Word Embedding)**과 밀집 표현, 대표적 임베딩 기법인 **Word2Vec / GloVe / FastText**, 파이토치의 **nn.Embedding()**과 **사전 훈련된 임베딩**, 그리고 문맥을 반영하는 **ELMo**까지 정리한다. 마지막에는 임베딩 층을 직접 사용하는 **단어 단위 RNN** 실습으로 마무리한다.

> **이 장의 흐름** : 원-핫 인코딩과 한계(차원·유사도) → 밀집 표현과 워드 임베딩 → Word2Vec(CBOW·Skip-gram·네거티브 샘플링) → GloVe·FastText → nn.Embedding()과 사전 훈련 임베딩 → ELMo(문맥 반영) → 단어 단위 RNN 실습.

> 이 장은 개념이 많지만, 핵심 메커니즘은 짧은 파이썬 코드로 직접 확인하며 넘어간다. 형태소 분석기(KoNLPy)나 대용량 사전 훈련 모델 다운로드가 필요한 부분은 코드 대신 설명으로 다루고, **다운로드 없이 그 자리에서 실행되는 장난감(toy) 예제**로 원-핫·코사인 유사도·룩업 테이블·임베딩 학습이 어떻게 동작하는지 손으로 따라가 본다.

# 1. 원-핫 인코딩 (One-hot Encoding)

컴퓨터는 문자보다 숫자를 더 잘 처리한다. 그래서 자연어 처리에서는 문자를 숫자로 바꾸는 여러 기법을 사용하는데, **원-핫 인코딩**은 그중 단어를 표현하는 **가장 기본적인 방법**이다. 머신 러닝·딥 러닝을 하려면 반드시 거쳐야 하는 표현 방법이므로 여기서 확실히 짚고 넘어간다.

## 1-1. 단어 집합(Vocabulary)과 정수 인코딩

**단어 집합(vocabulary)**은 **서로 다른 단어들의 집합**이다. 여기서 "서로 다른 단어"라는 정의에 주목할 필요가 있다. 단어 집합에서는 기본적으로 book과 books처럼 단어의 **변형 형태도 다른 단어로 간주**한다.

원-핫 인코딩을 위해 가장 먼저 할 일은 단어 집합을 만드는 것이다. 텍스트의 모든 단어를 중복 없이 모아 단어 집합을 만들고, 각 단어에 고유한 숫자를 부여하는 **정수 인코딩**을 진행한다. 예를 들어 텍스트에 단어가 총 5,000개라면 단어 집합의 크기는 5,000이고, 각 단어에 1번부터 5,000번까지 인덱스를 부여한다.

## 1-2. 원-핫 인코딩이란?

**원-핫 인코딩**은 단어 집합의 크기를 벡터의 차원으로 하고, 표현하고 싶은 단어의 인덱스에는 **1**을, 다른 인덱스에는 **0**을 부여하는 표현 방식이다. 이렇게 만들어진 벡터를 **원-핫 벡터(One-hot vector)**라고 한다. 두 단계로 정리하면 다음과 같다.

1. 각 단어에 고유한 정수 인덱스를 부여한다. (정수 인코딩)
2. 표현할 단어의 인덱스 위치에 1을, 나머지에는 0을 부여한다.

한국어 문장 `'나는 자연어 처리를 배운다'`로 원-핫 벡터를 만들어 본다. 원본 교안은 KoNLPy의 Okt 형태소 분석기로 토큰화하지만, 여기서는 그 **토큰화 결과를 직접 제공**해 동작에 집중한다.

In [ ]:
# 교안의 '나는 자연어 처리를 배운다' 예시를 사전 토큰화 결과로 재현한다.
# (원본은 KoNLPy Okt 형태소 분석기를 사용하지만, 여기서는 토큰화 결과를 직접 제공한다.)
token = ['나', '는', '자연어', '처리', '를', '배운다']

# 각 토큰에 고유한 정수 인덱스 부여 (정수 인코딩)
word2index = {}
for voca in token:
    if voca not in word2index:
        word2index[voca] = len(word2index)
print(word2index)

# 단어를 입력하면 해당 단어의 원-핫 벡터를 반환하는 함수
def one_hot_encoding(word, word2index):
    one_hot_vector = [0] * len(word2index)
    index = word2index[word]
    one_hot_vector[index] = 1
    return one_hot_vector

# '자연어'는 인덱스 2 -> 인덱스 2 자리만 1
print("'자연어' ->", one_hot_encoding('자연어', word2index))

## 1-3. 원-핫 인코딩의 한계: 차원과 유사도

원-핫 인코딩에는 두 가지 본질적 한계가 있다.

**한계 1 — 저장 공간의 비효율(차원의 폭증)** : 원-핫 벡터는 **단어 집합의 크기가 곧 벡터의 차원**이 된다. 단어가 1,000개인 코퍼스라면 모든 단어가 1,000차원 벡터가 되고, 그중 단 하나만 1이며 나머지 999개는 0이다. 단어가 늘어날수록 벡터 차원이 한없이 커져 저장 공간 측면에서 매우 비효율적이다.

**한계 2 — 단어 간 유사도를 표현하지 못함** : 늑대, 호랑이, 강아지, 고양이를 각각 `[1,0,0,0]`, `[0,1,0,0]`, `[0,0,1,0]`, `[0,0,0,1]`로 표현하면, 강아지와 늑대가 유사하고 호랑이와 고양이가 유사하다는 사실을 전혀 담을 수 없다. 모든 원-핫 벡터는 서로 **직교(orthogonal)**하기 때문이다.

이 한계는 검색 시스템 등에서 치명적이다. `'삿포로 숙소'`를 검색했을 때 `'삿포로 게스트 하우스'`, `'삿포로 호텔'` 같은 유사어 결과를 함께 보여주려면 단어 간 유사도를 계산할 수 있어야 한다. 원-핫 벡터끼리 **코사인 유사도**를 구해 보면 어떤 쌍을 골라도 값이 0임을 직접 확인한다.

In [ ]:
import torch

# 5개 단어를 원-핫 벡터로 표현한다 (dog, cat, computer, netbook, book)
dog      = torch.FloatTensor([1, 0, 0, 0, 0])
cat      = torch.FloatTensor([0, 1, 0, 0, 0])
computer = torch.FloatTensor([0, 0, 1, 0, 0])
netbook  = torch.FloatTensor([0, 0, 0, 1, 0])
book     = torch.FloatTensor([0, 0, 0, 0, 1])

# 사람 직관으로는 (강아지, 고양이)가 (고양이, 컴퓨터)보다 유사할 것 같지만,
# 어떤 두 단어를 골라 코사인 유사도를 구해도 전부 0이 나온다.
print('cos(dog, cat)          =', torch.cosine_similarity(dog, cat, dim=0).item())
print('cos(cat, computer)     =', torch.cosine_similarity(cat, computer, dim=0).item())
print('cos(computer, netbook) =', torch.cosine_similarity(computer, netbook, dim=0).item())
print('cos(netbook, book)     =', torch.cosine_similarity(netbook, book, dim=0).item())
# 원-핫 벡터는 서로 직교하므로 단어 간 의미적 유사도를 전혀 표현하지 못한다.

# 2. 워드 임베딩: 밀집 표현 (Word Embedding)

원-핫 인코딩의 한계를 넘기 위해 단어를 **밀집 벡터(dense vector)**로 표현하는 방법이 등장한다. 단어를 밀집 벡터로 표현하는 것을 **워드 임베딩(Word Embedding)**이라 하고, 그 결과로 나온 벡터를 **임베딩 벡터(embedding vector)**라고 한다.

## 2-1. 희소 표현 vs 밀집 표현

벡터나 행렬의 값이 대부분 0으로 표현되는 방법을 **희소 표현(sparse representation)**이라 한다. 원-핫 벡터가 바로 희소 벡터다. 단어 집합이 클수록 고차원이 되고, 단어 간 유사도를 표현하지 못한다는 단점이 있다.

이와 반대되는 표현이 **밀집 표현(dense representation)**이다. 밀집 표현은 벡터의 차원을 단어 집합의 크기로 두지 않고 **사용자가 설정한 값**으로 모든 단어의 차원을 맞춘다. 또한 0과 1만 갖는 것이 아니라 **실수값**을 가진다.

| 항목 | 원-핫(희소) 벡터 | 임베딩(밀집) 벡터 |
|:--|:--|:--|
| 예시 | `강아지 = [0,0,0,0,1,0, ... ,0]` (차원 10,000) | `강아지 = [0.2, 1.8, 1.1, -2.1, ...]` (차원 128) |
| 차원 | 고차원(단어 집합의 크기) | 저차원(사용자 지정) |
| 값의 타입 | 1과 0 | 실수 |

> 단어가 10,000개일 때 원-핫 벡터는 10,000차원이지만, 밀집 표현의 차원을 128로 정하면 모든 단어가 128차원 실수 벡터로 표현된다. 차원이 조밀해졌다고 하여 **밀집 벡터(dense vector)**라고 부른다.

## 2-2. 워드 임베딩과 임베딩 벡터

단어를 밀집 벡터의 형태로 표현하는 방법이 **워드 임베딩**이고, 그 결과 벡터가 **임베딩 벡터**다. 워드 임베딩 방법론으로는 **LSA, Word2Vec, FastText, GloVe** 등이 있다.

파이토치가 제공하는 도구인 **nn.Embedding()**은 위 방법들을 직접 사용하지는 않는다. 대신 단어를 **랜덤한 값을 가지는 밀집 벡터로 변환한 뒤, 인공 신경망의 가중치를 학습하듯 단어 벡터를 학습**하는 방식을 쓴다. 아래 표는 두 표현의 차이를 요약한다.

| | 원-핫 벡터 | 임베딩 벡터 |
|:--|:--|:--|
| 차원 | 고차원(단어 집합의 크기) | 저차원 |
| 다른 표현 | 희소 벡터의 일종 | 밀집 벡터의 일종 |
| 표현 방법 | 수동 | 훈련 데이터로부터 학습함 |
| 값의 타입 | 1과 0 | 실수 |

# 3. Word2Vec

원-핫 벡터는 단어 간 유사도를 계산할 수 없다. 단어의 **의미**를 반영해 벡터화하는 대표적 방법이 **Word2Vec**이다. Word2Vec 임베딩으로는 다음과 같은 단어 벡터 연산이 가능할 만큼 의미 관계가 벡터에 담긴다.

```
고양이 + 애교 = 강아지
한국 - 서울 + 도쿄 = 일본
```

이런 연산이 가능한 이유는 각 단어 벡터가 단어 간 유사도를 반영한 값을 가지기 때문이다.

## 3-1. 분산 표현과 분포 가설

Word2Vec의 바탕에는 **분산 표현(distributed representation)**이 있고, 분산 표현은 **분포 가설(distributional hypothesis)** 위에 만들어진다.

> **분포 가설** : "비슷한 위치(문맥)에서 등장하는 단어들은 비슷한 의미를 가진다."

예를 들어 '강아지'는 귀엽다, 예쁘다, 애교 같은 단어와 자주 함께 등장한다. 분포 가설에 따라 이런 텍스트를 벡터화하면 그 단어들은 의미적으로 가까운 벡터가 된다. 분산 표현은 단어의 의미를 **여러 차원에 분산**하여 저차원 밀집 벡터로 표현하므로, 단어 간 유사도를 계산할 수 있다.

요약하면, 희소 표현이 고차원에서 각 차원이 분리된 표현이었다면, **분산 표현은 저차원에 단어의 의미를 여러 차원으로 분산**한 표현이다.

## 3-2. CBOW (Continuous Bag of Words)

Word2Vec에는 **CBOW**와 **Skip-gram** 두 방식이 있다. CBOW는 **주변 단어로 중심 단어를 예측**하고, Skip-gram은 **중심 단어로 주변 단어를 예측**한다. 메커니즘이 거의 동일하므로 CBOW를 이해하면 Skip-gram도 쉽게 이해된다.

예문 `"The fat cat sat on the mat"`에서 `sat`을 예측한다고 하자. 예측 대상 `sat`을 **중심 단어(center word)**, 예측에 사용하는 단어들을 **주변 단어(context word)**라고 한다. 앞뒤로 몇 개를 볼지 정한 범위가 **윈도우(window)**다. 윈도우 크기가 $n$ 이면 참고하는 주변 단어 수는 $2n$ 이 된다.

윈도우를 한 칸씩 옮기며 (주변 단어, 중심 단어) 쌍을 만드는 방법을 **슬라이딩 윈도우(sliding window)**라 한다. 예문에 대해 윈도우 크기 2로 데이터셋을 직접 만들어 본다.

In [ ]:
# CBOW 학습 데이터 만들기: 슬라이딩 윈도우로 (주변 단어, 중심 단어) 쌍을 생성한다.
sentence = "The fat cat sat on the mat".split()

def build_cbow_dataset(words, window_size=2):
    dataset = []
    for center_idx in range(len(words)):
        context = []
        for w in range(-window_size, window_size + 1):
            ctx_idx = center_idx + w
            if w == 0 or ctx_idx < 0 or ctx_idx >= len(words):
                continue  # 중심 단어 자신과 문장 밖 인덱스는 제외
            context.append(words[ctx_idx])
        dataset.append((context, words[center_idx]))
    return dataset

for context, center in build_cbow_dataset(sentence, window_size=2):
    print(f'주변={context}  ->  중심={center}')

## 3-3. CBOW의 신경망 구조

CBOW는 입력층, **투사층(projection layer)**, 출력층으로 구성된 얕은 신경망이다. 은닉층이 하나뿐이고 활성화 함수가 없으므로 **심층 신경망이 아니라 얕은 신경망(Shallow Neural Network)**이다. 핵심은 두 가지다.

- **투사층의 크기 $M$ 이 곧 임베딩 벡터의 차원**이다.
- 입력층-투사층 가중치 $W$ 는 $V \times M$, 투사층-출력층 가중치 $W'$ 는 $M \times V$ 행렬이다. 여기서 $V$ 는 단어 집합의 크기다. 이 둘은 전치 관계가 아니라 **서로 다른 행렬**이다.

**룩업 테이블(lookup table)** : $i$ 번째만 1인 원-핫 벡터와 $W$ 의 곱은 사실 $W$ 의 $i$ 번째 행을 그대로 읽어오는 것과 같다. 그래서 이 연산을 룩업 테이블이라 하며, 학습이 끝난 뒤 $W$ 의 각 행벡터가 곧 각 단어의 $M$ 차원 임베딩 벡터가 된다.

주변 단어들의 벡터는 투사층에서 **평균**되고($\frac{V_{fat}+V_{cat}+V_{on}+V_{the}}{2n}$), 이 평균 벡터에 $W'$ 를 곱한 뒤 **소프트맥스**를 취해 **스코어 벡터**를 얻는다. 스코어 벡터의 $j$ 번째 값은 $j$ 번째 단어가 중심 단어일 확률을 뜻한다. 이 스코어 벡터를 실제 중심 단어의 원-핫 벡터에 가깝게 만들기 위해 손실 함수로 **cross-entropy**를 사용한다.

$$ H(\hat{y}, y) = -\sum_{j=1}^{|V|} y_j \log(\hat{y}_j) $$

$y$ 가 원-핫 벡터이므로 중심 단어 인덱스 $c$ 만 살아남아 $H(\hat{y}, y) = -\log(\hat{y}_c)$ 로 간소화된다. 정확히 예측하면($\hat{y}_c = 1$) 손실이 $-\log(1)=0$ 이 되므로, 이 값을 최소화하는 방향으로 $W$ 와 $W'$ 를 역전파로 학습한다.

## 3-4. Skip-gram

Skip-gram은 **중심 단어 하나로 주변 단어들을 예측**한다. 입력이 중심 단어 하나뿐이므로 CBOW와 달리 **투사층에서 벡터의 평균을 구하지 않는다**. 여러 논문의 성능 비교에서 전반적으로 **Skip-gram이 CBOW보다 성능이 좋다**고 알려져 있다.

## 3-5. 네거티브 샘플링 (Negative Sampling, SGNS)

대체로 Word2Vec을 쓴다고 하면 **SGNS(Skip-Gram with Negative Sampling)**를 사용한다. 동기는 **속도**다. 출력층의 소프트맥스는 단어 집합 크기만큼의 모든 값을 계산하고, 역전파로 **모든 단어의 임베딩을 매번 조정**한다. 단어 집합이 수백만이면 이는 매우 무거운 작업이다.

핵심 아이디어는, 전체 단어 집합 대신 **일부만 보고 마지막 단계를 이진 분류로 바꾸는 것**이다. '강아지', '고양이', '애교' 같은 실제 주변 단어를 **긍정(positive)**으로 두고, '돈가스', '컴퓨터'처럼 무관한 단어를 랜덤으로 뽑아 **부정(negative)**으로 둔 뒤, "이 단어가 실제 주변 단어인가?"를 맞히는 이진 분류를 수행한다. 다중 클래스 분류를 이진 분류로 바꾸면서 연산량이 훨씬 효율적이 된다.

## 3-6. gensim으로 Word2Vec 학습하기

파이썬의 **gensim** 패키지에는 Word2Vec이 구현돼 있어 손쉽게 단어를 임베딩 벡터로 변환할 수 있다. 주요 하이퍼파라미터는 다음과 같다.

- `vector_size` : 임베딩 벡터의 차원
- `window` : 컨텍스트 윈도우 크기
- `min_count` : 단어 최소 빈도 수 제한(빈도가 적은 단어는 학습 제외)
- `workers` : 학습에 사용할 프로세스 수
- `sg` : 0이면 CBOW, 1이면 Skip-gram

> 실제 교안은 영어 TED 자막(약 27만 문장)이나 한국어 NSMC 영화 리뷰(약 20만 개)를 다운로드해 학습하고, `model.wv.most_similar('man')` 으로 유사 단어를 확인한다. 여기서는 다운로드 없이 동작을 확인하기 위해 **의도적으로 의미가 가까운 단어들을 같은 문맥에 배치한 장난감 코퍼스**로 같은 API를 실행해 본다.

In [ ]:
from gensim.models import Word2Vec

# 다운로드 없이 동작을 확인하기 위한 장난감 코퍼스
# (의미가 가까운 단어들을 같은 문맥에 함께 배치한다 -> 분포 가설)
corpus = [
    ['왕', '여왕', '남자', '여자'],
    ['왕', '남자'], ['여왕', '여자'],
    ['고양이', '강아지', '동물'],
    ['고양이', '동물'], ['강아지', '동물'],
] * 50

# sg=0 -> CBOW, vector_size=임베딩 차원, window=문맥 크기, min_count=최소 빈도
model = Word2Vec(sentences=corpus, vector_size=30, window=2,
                 min_count=1, workers=1, sg=0, seed=0)

# 완성된 임베딩 행렬의 크기 (단어 수, 차원)
print('임베딩 행렬 크기:', model.wv.vectors.shape)
# 입력 단어와 가장 유사한 단어들 출력
print("'고양이'와 유사한 단어:", model.wv.most_similar('고양이', topn=3))

# 4. GloVe와 FastText

이 절에서는 Word2Vec의 대안으로 사용되는 **GloVe**와 **FastText**를 다룬다. GloVe는 카운트 기반과 예측 기반을 결합하고, FastText는 단어 내부의 **서브워드(subword)**를 고려한다.

## 4-1. GloVe: 카운트 + 예측

**GloVe(Global Vectors for Word Representation)**는 2014년 스탠퍼드대에서 개발한 임베딩 방법론으로, **카운트 기반(LSA)**과 **예측 기반(Word2Vec)**을 모두 사용한다.

- **LSA**는 전체 통계 정보를 입력으로 받아 차원을 축소하지만, 단어 유추 작업(왕:남자 = 여왕:?)에 약하다.
- **Word2Vec**은 유추 작업에는 강하지만, 윈도우 내 주변 단어만 보므로 **코퍼스의 전체 통계 정보를 반영하지 못한다.**

GloVe는 이 두 한계를 모두 지적하며, 카운트 기반과 예측 기반을 결합한다. 출발점은 **윈도우 기반 동시 등장 행렬(Co-occurrence Matrix)**이다. 이 행렬은 행과 열을 단어 집합으로 두고, $i$ 단어의 윈도우 내에서 $k$ 단어가 등장한 횟수를 $i$ 행 $k$ 열에 적는다. 동시 등장 행렬을 직접 만들어 본다.

In [ ]:
import numpy as np

# 교안 예시: 윈도우 크기 1로 동시 등장 행렬을 만든다.
corpus = [
    "I like deep learning",
    "I like NLP",
    "I enjoy flying",
]
tokenized = [s.split() for s in corpus]

# 단어 집합 (첫 등장 순서 유지)
vocab = []
for s in tokenized:
    for w in s:
        if w not in vocab:
            vocab.append(w)
idx = {w: i for i, w in enumerate(vocab)}

# 동시 등장 행렬 (window=1: 좌우 1칸씩 참고)
window = 1
M = np.zeros((len(vocab), len(vocab)), dtype=int)
for s in tokenized:
    for i, w in enumerate(s):
        for j in range(max(0, i - window), min(len(s), i + window + 1)):
            if i == j:
                continue
            M[idx[w], idx[s[j]]] += 1

print('단어 집합:', vocab)
print(M)
# i 주변의 k 빈도 == k 주변의 i 빈도 이므로 행렬은 전치해도 동일하다(대칭).
print('대칭 행렬 여부:', np.array_equal(M, M.T))

## 4-2. 동시 등장 확률과 손실 함수

**동시 등장 확률** $P(k \mid i)$ 는 중심 단어 $i$ 가 등장했을 때 주변 단어 $k$ 가 등장할 조건부 확률이다(동시 등장 행렬에서 $i$ 행의 합을 분모, $i$ 행 $k$ 열을 분자로 한 값). GloVe 논문의 유명한 예를 보면 직관이 잡힌다.

| 비(ratio) | k=solid | k=gas | k=water | k=fashion |
|:--|:--:|:--:|:--:|:--:|
| $P(k \mid ice)$ | 큰 값 | 작은 값 | 큰 값 | 작은 값 |
| $P(k \mid steam)$ | 작은 값 | 큰 값 | 큰 값 | 작은 값 |
| $P(k \mid ice) / P(k \mid steam)$ | 큰 값(≈8.9) | 작은 값(≈0.085) | ≈1 | ≈1 |

solid는 ice와 더 자주 등장하므로 비가 1보다 크고, gas는 steam과 더 자주 등장하므로 비가 1보다 작다. water처럼 둘 다와 자주 등장하거나 fashion처럼 둘 다와 드물게 등장하면 비가 1에 가깝다.

> **GloVe의 아이디어(한 줄 요약)** : "임베딩된 중심 단어와 주변 단어 벡터의 **내적**이 코퍼스 전체의 **동시 등장 확률(의 로그)**이 되도록 만든다."

$$ \text{dot}(w_i,\ \tilde{w}_k) \approx \log P(k \mid i) = \log X_{ik} - \log X_i $$

준동형(Homomorphism) 조건을 만족시키며 식을 전개하면, $\log X_i$ 항을 편향 $b_i,\ \tilde{b}_k$ 로 대체해 손실 함수의 핵심 식 $w_i^T \tilde{w}_k + b_i + \tilde{b}_k = \log X_{ik}$ 를 얻는다. 마지막으로 동시 등장 행렬이 희소 행렬이라는 점을 고려해, 빈도가 낮은 쌍의 영향을 줄이는 **가중치 함수** $f(x) = \min(1,\ (x/x_{max})^{3/4})$ 를 곱해 최종 손실 함수를 완성한다.

$$ \text{Loss} = \sum_{m,n=1}^{V} f(X_{mn})\left(w_m^T \tilde{w}_n + b_m + \tilde{b}_n - \log X_{mn}\right)^2 $$

## 4-3. FastText: 서브워드 (Subword)

**FastText**는 페이스북에서 개발한, Word2Vec의 확장이다. 가장 큰 차이는 단어를 바라보는 단위다. Word2Vec은 단어를 **쪼갤 수 없는 단위**로 보지만, FastText는 하나의 단어 안에 여러 **내부 단어(서브워드, subword)**가 있다고 본다.

FastText는 각 단어를 글자 단위 **n-gram**으로 취급한다. 예를 들어 $n=3$ 일 때 `apple`은 시작/끝 기호 `<`, `>`를 붙여 분해한다. 실제로는 $n$ 의 최소·최대 범위(기본 3~6)의 모든 n-gram에 **단어 전체 토큰** `<apple>`까지 더해 벡터화하고, 단어 `apple`의 벡터는 이 **서브워드 벡터들의 합**으로 구성한다. 서브워드 분해를 직접 만들어 본다.

In [ ]:
# FastText는 단어를 글자 단위 n-gram(서브워드)으로 분해한다.
def char_ngrams(word, n):
    w = '<' + word + '>'                       # 시작/끝 기호 추가
    return [w[i:i+n] for i in range(len(w) - n + 1)]

word = 'apple'
print('n=3      :', char_ngrams(word, 3))

# 실제로는 최소~최대 범위(기본 3~6)의 모든 n-gram + 단어 전체 토큰을 함께 사용한다.
subwords = []
for n in range(3, 7):
    subwords += char_ngrams(word, n)
subwords.append('<' + word + '>')             # 단어 전체 토큰
print('n=3~6 +전체:', subwords)

## 4-4. FastText의 강점: OOV와 희귀 단어

서브워드를 학습하면 Word2Vec에는 없던 강점이 생긴다.

1. **모르는 단어(OOV) 대응** : 데이터셋이 충분하면, 학습하지 않은 단어라도 그 단어의 n-gram이 다른 단어의 n-gram과 겹칠 때 임베딩을 계산할 수 있다. 예를 들어 `birthplace`를 학습한 적 없어도 `birth`와 `place`라는 내부 단어를 봤다면 벡터를 얻는다.
2. **희귀 단어(Rare word)와 오타에 강함** : 등장 빈도가 적은 단어, 심지어 오타(`apple` → `appple`)도 다수의 동일한 n-gram을 공유하므로 일정 수준의 임베딩 성능을 보인다.

gensim으로 작은 코퍼스를 학습해, Word2Vec은 OOV 단어에서 에러가 나지만 FastText는 유사 단어를 계산해내는 차이를 직접 확인한다.

In [ ]:
from gensim.models import Word2Vec, FastText

corpus = [
    ['king', 'queen', 'man', 'woman'],
    ['king', 'man'], ['queen', 'woman'],
    ['cat', 'dog', 'animal'],
    ['cat', 'animal'], ['dog', 'animal'],
] * 50

w2v = Word2Vec(corpus, vector_size=30, window=2, min_count=1, workers=1, sg=1, seed=0)
ft  = FastText(corpus, vector_size=30, window=2, min_count=1, workers=1, sg=1,
               seed=0, min_n=2, max_n=4)

oov = 'kingg'   # 학습 데이터에 없는 단어(king의 오타)

# Word2Vec: 학습하지 않은 단어는 단어 집합에 없어 유사도를 구할 수 없다 -> KeyError
try:
    w2v.wv.most_similar(oov)
except KeyError as e:
    print('Word2Vec ->', e)

# FastText: 서브워드(king 등)를 공유하므로 OOV 단어도 임베딩을 계산할 수 있다
print('FastText ->', ft.wv.most_similar(oov, topn=3))

## 4-5. 한국어에서의 FastText

한국어도 OOV 문제 완화를 위해 FastText를 적용하려는 시도가 있었다.

- **음절 단위** : `'자연어처리'`를 $n=3$ 으로 n-gram 하면 `<자연, 자연어, 연어처, 어처리, 처리>` 처럼 분해된다.
- **자모 단위** : 초성·중성·종성으로 분리하고(종성이 없으면 `_` 토큰 사용) n-gram을 적용한다. `'자연어처리'`는 `ㅈ ㅏ _ ㅇ ㅕ ㄴ ㅇ ㅓ _ ㅊ ㅓ _ ㄹ ㅣ _` 로 분리된 뒤 `< ㅈ ㅏ, ㅈ ㅏ _, ㅏ _ ㅇ, ...>` 처럼 n-gram이 만들어진다.

자모 단위로 가면 오타·노이즈에 더 강한 임베딩을 기대할 수 있다.

# 5. 파이토치의 nn.Embedding()

파이토치에서 임베딩 벡터를 사용하는 방법은 크게 두 가지다. (1) **임베딩 층(embedding layer)**을 만들어 훈련 데이터로부터 처음부터 학습하는 방법, (2) **사전 훈련된 임베딩(pre-trained word embedding)**을 가져와 사용하는 방법이다. 먼저 전자를 다룬다.

## 5-1. 임베딩 층은 룩업 테이블이다

임베딩 층의 입력으로 쓰려면 각 단어가 **정수 인코딩**되어 있어야 한다. 동작 흐름은 다음과 같다.

```
어떤 단어 -> 고유한 정수 -> 임베딩 층 통과 -> 밀집 벡터(임베딩 벡터)
```

임베딩 층은 입력 정수를 밀집 벡터로 맵핑하고, 이 벡터는 신경망 학습 과정에서 가중치처럼 갱신된다. 본질은 **특정 단어 인덱스로 테이블에서 해당 행을 꺼내오는 룩업 테이블**이다. 헷갈리기 쉬운 점은, 파이토치는 단어를 **원-핫 벡터로 바꾸지 않고 정수 인덱스만으로** 임베딩 층의 입력으로 사용한다는 것이다. nn.Embedding() 없이 룩업 테이블을 직접 구현해 이해해 본다.

In [ ]:
import torch

train_data = 'you need to know how to code'
# 중복을 제거한 단어 집합 생성
word_set = set(train_data.split())
# 각 단어에 고유 정수 부여 (<unk>=0, <pad>=1 은 따로 예약)
vocab = {word: i + 2 for i, word in enumerate(word_set)}
vocab['<unk>'] = 0
vocab['<pad>'] = 1
print(vocab)

# 단어 집합 크기(8) x 임베딩 차원(3) 크기의 임베딩 테이블을 직접 만든다 (값은 예시용).
embedding_table = torch.FloatTensor([
    [0.0, 0.0, 0.0],   # 0: <unk>
    [0.0, 0.0, 0.0],   # 1: <pad>
    [0.2, 0.9, 0.3],
    [0.1, 0.5, 0.7],
    [0.2, 0.1, 0.8],
    [0.4, 0.1, 0.1],
    [0.1, 0.8, 0.9],
    [0.6, 0.1, 0.1],
])

sample = 'you need to run'.split()
# 각 단어를 정수로 변환 (단어 집합에 없는 'run'은 <unk>=0 으로 대체)
idxes = torch.LongTensor([vocab.get(w, vocab['<unk>']) for w in sample])

# 정수 인덱스로 테이블의 해당 행을 그대로 읽어온다 = 룩업 테이블
lookup_result = embedding_table[idxes, :]
print(lookup_result)

## 5-2. nn.Embedding() 사용하기

이제 nn.Embedding()으로 **학습 가능한** 임베딩 테이블을 만든다. 주요 인자는 다음과 같다.

- `num_embeddings` : 임베딩할 단어들의 개수, 즉 **단어 집합의 크기**
- `embedding_dim` : 임베딩 벡터의 차원(사용자가 정하는 하이퍼파라미터)
- `padding_idx` : (선택) 패딩 토큰의 인덱스. 해당 행은 0 벡터로 고정된다.

In [ ]:
import torch.nn as nn

train_data = 'you need to know how to code'
word_set = set(train_data.split())
vocab = {tkn: i + 2 for i, tkn in enumerate(word_set)}
vocab['<unk>'] = 0
vocab['<pad>'] = 1

# num_embeddings=단어 집합 크기, embedding_dim=임베딩 차원, padding_idx=패딩 토큰 인덱스
embedding_layer = nn.Embedding(num_embeddings=len(vocab),
                               embedding_dim=3,
                               padding_idx=1)
print(embedding_layer.weight)
# padding_idx(=1) 행은 0 벡터로 고정되고, 나머지 행은 학습 과정에서 갱신된다.

## 5-3. 사전 훈련된 워드 임베딩 (Pre-trained Word Embedding)

훈련 데이터가 부족하면, 작업에 특화된 임베딩을 처음부터 학습하기보다 **이미 방대한 데이터로 학습된 임베딩(Word2Vec, GloVe 등)**을 가져와 쓰는 편이 성능에 유리할 때가 많다.

흐름을 비교하기 위해, 먼저 **사전 훈련 임베딩 없이** nn.Embedding()을 처음부터 학습하는 감성 분류 모델을 만든다. 긍정 문장은 레이블 1, 부정 문장은 0이다. 모델은 `임베딩 -> 평탄화(Flatten) -> 선형 -> 시그모이드` 구조의 이진 분류기이며, 손실 함수로는 **바이너리 크로스엔트로피(nn.BCELoss)**를 쓴다.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter

# 1) 데이터: 긍정(1) / 부정(0)
sentences = ['nice great best amazing', 'stop lies', 'pitiful nerd', 'excellent work',
             'supreme quality', 'bad', 'highly respectable']
y_train = [1, 0, 0, 1, 1, 0, 1]
tokenized = [s.split() for s in sentences]

# 2) 단어 집합 (빈도순) + 패딩/UNK 토큰
counts = Counter(w for s in tokenized for w in s)
vocab = sorted(counts, key=counts.get, reverse=True)
word_to_index = {'<PAD>': 0, '<UNK>': 1}
for i, w in enumerate(vocab):
    word_to_index[w] = i + 2
vocab_size = len(word_to_index)

# 3) 정수 인코딩 + 패딩 (최대 길이에 맞춰 0으로 채움)
encoded = [[word_to_index.get(w, 1) for w in s] for s in tokenized]
max_len = max(len(s) for s in encoded)
X_train = np.zeros((len(encoded), max_len), dtype=int)
for i, s in enumerate(encoded):
    X_train[i, :len(s)] = np.array(s)
y_train = np.array(y_train)

# 4) 모델: 임베딩 -> 평탄화 -> 선형 -> 시그모이드 (이진 분류)
class SimpleModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(embedding_dim * max_len, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.fc(self.flatten(self.embedding(x))))

model = SimpleModel(vocab_size, embedding_dim=100)
criterion = nn.BCELoss()                 # 이진 분류 -> 바이너리 크로스엔트로피
optimizer = Adam(model.parameters())

ds = TensorDataset(torch.tensor(X_train, dtype=torch.long),
                   torch.tensor(y_train, dtype=torch.float32))
dl = DataLoader(ds, batch_size=2)

for epoch in range(10):
    for inputs, targets in dl:
        optimizer.zero_grad()
        outputs = model(inputs).view(-1)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1:2d}, Loss: {loss.item():.4f}')

### 사전 훈련된 임베딩을 주입하는 방법

사전 훈련된 임베딩(예: 구글이 공개한 300차원 Word2Vec, 단어 300만 개)을 쓰려면, gensim으로 모델을 로드한 뒤 **단어 집합 크기 × 300 크기의 임베딩 행렬을 0으로 초기화**하고, 단어별로 사전 훈련 벡터를 채워 넣는다. 그 행렬을 nn.Embedding의 가중치로 주입한다. 핵심 코드 패턴만 보면 다음과 같다(대용량 다운로드가 필요하므로 여기서는 실행하지 않는다).

```python
import gensim
# word2vec_model = gensim.models.KeyedVectors.load_word2vec_format(
#     'GoogleNews-vectors-negative300.bin.gz', binary=True)

embedding_matrix = np.zeros((vocab_size, 300))         # 0으로 초기화
for word, i in word_to_index.items():
    if i > 2 and word in word2vec_model:               # PAD/UNK 제외, 있는 단어만
        embedding_matrix[i] = word2vec_model[word]

# nn.Embedding에 사전 훈련 가중치를 주입
self.embedding = nn.Embedding(vocab_size, 300)
self.embedding.weight = nn.Parameter(torch.tensor(embedding_matrix, dtype=torch.float32))
self.embedding.weight.requires_grad = True             # 필요 시 미세 조정 허용
```

`<PAD>`(0번), `<UNK>`(1번)는 실제 단어가 아니므로 0 벡터로 남겨 둔다. 이렇게 주입한 임베딩 행렬의 특정 단어 벡터가 원본 사전 훈련 벡터와 일치하는지 `np.all(word2vec_model['great'] == embedding_matrix[idx])` 로 검증할 수 있다.

# 6. ELMo: 문맥을 반영한 임베딩

**ELMo(Embeddings from Language Model)**는 2018년에 제안된 워드 임베딩 방법론이다. 가장 큰 특징은 **사전 훈련된 언어 모델(Pre-trained language model)**을 사용한다는 점이며, 이름에 LM이 들어간 이유다.

## 6-1. 문맥을 반영한 워드 임베딩의 필요성

`Bank Account`(은행 계좌)와 `River Bank`(강둑)에서 `Bank`는 전혀 다른 의미를 갖는다. 그런데 Word2Vec이나 GloVe는 `Bank`를 항상 **같은 벡터**(예: `[0.2, 0.8, -1.2]`)로 표현해 이 차이를 담지 못한다.

같은 표기라도 **문맥에 따라 다르게 임베딩**할 수 있으면 성능을 올릴 수 있다. 이렇게 문맥을 고려한 임베딩을 **문맥을 반영한 워드 임베딩(Contextualized Word Embedding)**이라고 하며, ELMo가 그 대표다.

## 6-2. biLM(Bidirectional Language Model)의 사전 훈련

ELMo는 다음 단어를 예측하는 **순방향 RNN 언어 모델**뿐 아니라, 반대 방향으로 문장을 스캔하는 **역방향 RNN 언어 모델**도 함께 학습한다. 양쪽 방향을 모두 학습·활용하므로 이를 **biLM(Bidirectional Language Model)**이라 한다. biLM은 기본적으로 **은닉층이 최소 2개 이상인 다층 구조**를 전제로 한다.

> **양방향 RNN과 biLM은 다르다.** 양방향 RNN은 순방향·역방향 은닉 상태를 **연결(concatenate)해 다음 층의 입력**으로 쓴다. 반면 biLM은 순방향 언어 모델과 역방향 언어 모델을 **별개의 모델**로 보고 학습한다.

참고로 biLM의 각 시점 입력 단어 벡터는 임베딩 층이 아니라 **합성곱 신경망 기반의 문자 임베딩(character embedding)**으로 얻는다. 문자 임베딩은 서브워드처럼 `dog`와 `doggy`의 연관성을 문맥과 무관하게 찾아내고 OOV에도 견고하다.

## 6-3. biLM의 활용: ELMo 표현 만들기

사전 훈련된 biLM으로 특정 단어를 임베딩하는 과정은 다음과 같다. 한 단어(time step)에 대해 biLM의 **각 층의 출력값**(첫째는 임베딩 층, 나머지는 각 은닉 상태)을 재료로 사용한다.

1. 각 층에서 순방향·역방향 출력값을 **연결(concatenate)**한다.
2. 각 층의 출력값에 **가중치** $s_1, s_2, s_3$ 를 준다.
3. 가중치를 곱한 층들을 **모두 더한다**. (2)~(3)을 합쳐 **가중합(Weighted Sum)**이라 한다.
4. 벡터 크기를 정하는 **스칼라 매개변수** $\gamma$ 를 곱한다.

이렇게 완성된 벡터가 **ELMo 표현(representation)**이다. ELMo 표현은 기존 임베딩(GloVe 등)과 **연결(concatenate)**해 downstream 작업(텍스트 분류, 질의 응답 등)의 입력으로 쓴다. 이때 **biLM의 가중치는 고정**하고, $s_1, s_2, s_3$ 와 $\gamma$ 는 작업을 학습하며 함께 학습된다. ELMo 표현을 만드는 가중합 과정을 코드로 시뮬레이션해 본다.

In [ ]:
import torch
torch.manual_seed(0)

# ELMo 표현 만들기 (개념 시뮬레이션)
# biLM의 은닉층이 2개일 때, 한 단어에 대해 3개 층의 출력이 나온다고 하자.
# (임베딩 층 1개 + 은닉층 2개). 각 층은 순방향|역방향을 연결(concatenate)한 벡터다.
dim = 4
layer0 = torch.cat([torch.randn(dim), torch.randn(dim)])   # 임베딩 층 (순방향|역방향)
layer1 = torch.cat([torch.randn(dim), torch.randn(dim)])   # 은닉층 1
layer2 = torch.cat([torch.randn(dim), torch.randn(dim)])   # 은닉층 2
layers = torch.stack([layer0, layer1, layer2])             # (층 개수=3, 2*dim)

# 1)~2) 각 층에 가중치 s 를 준다 (softmax로 정규화 -> 합이 1)
s = torch.softmax(torch.tensor([0.2, 0.5, 0.3]), dim=0)
# 3) 가중합(weighted sum)
weighted_sum = (s.unsqueeze(1) * layers).sum(dim=0)
# 4) 스칼라 매개변수 gamma 를 곱한다
gamma = torch.tensor(2.0)
elmo = gamma * weighted_sum

print('각 층 출력 크기 :', tuple(layers.shape))
print('가중치 s (합=1) :', [round(v, 3) for v in s.tolist()])
print('ELMo 표현 차원 :', tuple(elmo.shape))
# s 와 gamma 는 downstream 작업을 학습할 때 함께 학습되는 값이다.

# 7. 단어 단위 RNN 실습 (임베딩 사용)

마지막으로 임베딩 층을 직접 사용하는 실습을 한다. 이전 RNN 챕터의 **문자 단위 RNN**과 달리, 입력 단위를 **단어 단위**로 사용하고 파이토치의 **nn.Embedding()**을 RNN 앞단에 둔다. 목표는 `'Repeat is the best medicine for'`를 입력받아 `'is the best medicine for memory'`를 출력하는 모델이다.

## 7-1. 데이터 전처리하기

임의의 문장으로 단어장을 만들고, 각 단어에 고유한 정수 인덱스를 부여한다. 모르는 단어를 위한 `<unk>` 토큰도 추가한다. 그 뒤 데이터를 정수로 인코딩하면서, 입력 시퀀스와 레이블 시퀀스를 **한 칸 밀어서** 만든다(다음 단어 예측이므로 레이블은 입력을 한 칸 이동시킨 것).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

sentence = "Repeat is the best medicine for memory".split()
vocab = list(set(sentence))

# 단어 -> 정수 (1부터 부여), 모르는 단어 <unk>=0
word2index = {tkn: i for i, tkn in enumerate(vocab, 1)}
word2index['<unk>'] = 0
# 정수 -> 단어 (예측 결과를 단어로 되돌리기 위함)
index2word = {v: k for k, v in word2index.items()}

def build_data(sentence, word2index):
    encoded = [word2index[t] for t in sentence]            # 각 단어를 정수로 변환
    input_seq, label_seq = encoded[:-1], encoded[1:]       # 입력/레이블을 한 칸 밀어 분리
    input_seq = torch.LongTensor(input_seq).unsqueeze(0)   # 배치 차원 추가
    label_seq = torch.LongTensor(label_seq).unsqueeze(0)
    return input_seq, label_seq

X, Y = build_data(sentence, word2index)
print('입력  X:', X)   # Repeat is the best medicine for
print('레이블 Y:', Y)  # is the best medicine for memory

## 7-2. 모델 구현하기

이전 모델과 달라진 점은 **임베딩 층을 추가**했다는 것이다. `nn.Embedding(단어장 크기, 임베딩 차원)`으로 정수 시퀀스를 임베딩 벡터로 바꾼 뒤, RNN과 선형 층을 통과시킨다. 최종 출력 차원은 **단어장 크기**여야 다음 단어를 분류할 수 있다.

크기 변화를 따라가면 다음과 같다. `(배치, 시퀀스)` → (임베딩) → `(배치, 시퀀스, 임베딩 차원)` → (RNN) → `(배치, 시퀀스, 은닉 크기)` → (선형) → `(배치, 시퀀스, 단어장 크기)` → (view) → `(배치×시퀀스, 단어장 크기)`.

In [ ]:
class Net(nn.Module):
    def __init__(self, vocab_size, input_size, hidden_size, batch_first=True):
        super().__init__()
        self.embedding_layer = nn.Embedding(num_embeddings=vocab_size,    # 워드 임베딩
                                            embedding_dim=input_size)
        self.rnn_layer = nn.RNN(input_size, hidden_size,                  # 입력/은닉 차원
                                batch_first=batch_first)
        self.linear = nn.Linear(hidden_size, vocab_size)                  # 출력=단어장 크기

    def forward(self, x):
        # (배치, 시퀀스) -> (배치, 시퀀스, 임베딩 차원)
        output = self.embedding_layer(x)
        # -> output (배치, 시퀀스, 은닉 크기)
        output, hidden = self.rnn_layer(output)
        # -> (배치, 시퀀스, 단어장 크기)
        output = self.linear(output)
        # -> (배치*시퀀스, 단어장 크기)
        return output.view(-1, output.size(2))

# 하이퍼파라미터
vocab_size = len(word2index)   # 단어장 크기 (<unk> 포함)
input_size = 5                 # 임베딩 차원 = RNN 입력 차원
hidden_size = 20               # RNN 은닉 크기

model = Net(vocab_size, input_size, hidden_size, batch_first=True)

# 가중치가 랜덤 초기화된 상태로 한 번 예측해 출력 크기만 확인한다.
output = model(X)
print('예측 출력 크기:', tuple(output.shape))   # (시퀀스 길이, 단어장 크기)

## 7-3. 학습 및 예측

손실 함수로 **nn.CrossEntropyLoss**를 사용한다. 이 함수는 **내부에 소프트맥스를 포함**하므로 모델의 마지막 선형 층에 따로 소프트맥스를 적용하지 않으며, 실제 레이블도 **원-핫 인코딩이 불필요**하다(정수 인덱스를 그대로 받는다). 약 200 에포크 학습하면서 예측 문장이 정답에 수렴하는 과정을 확인한다.

In [ ]:
loss_function = nn.CrossEntropyLoss()   # 소프트맥스 포함 -> 레이블 원-핫 인코딩 불필요
optimizer = optim.Adam(params=model.parameters())

# 정수 시퀀스를 다시 단어 시퀀스로 되돌리는 함수
decode = lambda y: [index2word.get(x) for x in y]

for step in range(201):
    optimizer.zero_grad()
    output = model(X)
    loss = loss_function(output, Y.view(-1))   # 출력은 로짓, 레이블은 정수 인덱스
    loss.backward()
    optimizer.step()

    if step % 40 == 0:
        pred = output.softmax(-1).argmax(-1).tolist()       # 예측 단어 인덱스
        print(f'[{step+1:03d}/201] loss={loss:.4f}  ->  ' + ' '.join(['Repeat'] + decode(pred)))

---

이로써 12장 **텍스트 임베딩**의 핵심을 정리했다. 단어를 벡터로 표현하는 가장 기본인 **원-핫 인코딩**과 그 한계(차원 폭증·유사도 0)에서 출발해, 이를 극복하는 **밀집 표현과 워드 임베딩**, 의미 관계를 학습하는 **Word2Vec(CBOW·Skip-gram·네거티브 샘플링)**, 카운트와 예측을 결합한 **GloVe**, 서브워드로 OOV에 대응하는 **FastText**, 파이토치의 **nn.Embedding()**과 **사전 훈련 임베딩**, 문맥을 반영하는 **ELMo**, 그리고 임베딩 층을 직접 쓰는 **단어 단위 RNN 실습**까지 살펴봤다.

핵심은 **단어를 저차원 밀집 벡터로 학습하면 의미적 유사도를 계산할 수 있고, 이 임베딩이 이후 모든 NLP 모델의 입력층이 된다**는 것이다. 다음 장부터는 이 임베딩을 입력으로 받는 RNN·CNN 기반 텍스트 분류로 나아간다.